# Fashion Retrieval — Colab: kısmi fine-tune (Adım 5) — tohum tekrarı

**Ne yapar:** SigLIP görüntü kulesinin son 4 bloğu + projeksiyon başlığı, train_fit kırpıntılarından InfoNCE ile eğitilir.
Her tur sonunda val_internal Recall@1 ölçülür; en iyi model Drive'a yazılır (`outputs/checkpoints/ft/`). Bitince en iyi modelle
val_internal ve test öznitelikleri Drive'a (`features/siglip_vitb16+ft/`) çıkarılır ve **çalışma zamanı kendiliğinden kapanır**.

**Tohum tekrarı:** `SEEDS` listesindeki her tohum için eğitim baştan yapılır (tohum 0 daha önce yapıldı → `ft/`). Çıktılar
`outputs/checkpoints/ft_s<seed>/` ve `features/siglip_vitb16+ft_s<seed>/`. Tohum, başlangıç ağırlıklarını, ürünlerin batch sırasını,
dropout'u ve veri çoğaltmayı değiştirir; eğitim verisi, mimari ve ayarlar aynı kalır.

**Tahmini süre (L4):** zip açma 5 dk + tohum başına (8 tur × 2,2 dk + çıkarma 3 dk ≈ 21 dk) → 2 tohum ≈ **50 dk**.

**Nasıl kullanılır:** Çalışma zamanı → Çalışma zamanı türünü değiştir → **L4 GPU**. Sonra **Çalışma zamanı → Tümünü çalıştır**
(Ctrl+F9) deyip bırakın. Drive bağlama penceresi ilk hücrede bir kez izin ister; onu onayladıktan sonra başka müdahale gerekmez.
Oturum düşerse aynı şekilde yeniden "Tümünü çalıştır": eğitim kaldığı turdan sürer (last.pt Drive'da).

In [ ]:
GITHUB_USER = "ethembelka"
REPO_NAME   = "fashion-product-retrieval"
DRIVE_ROOT  = "/content/drive/MyDrive/CMPE9103_Retrieval"
CROPS_ZIP   = f"{DRIVE_ROOT}/data/crops.zip"
PROCESSED   = "/content/processed"
SEEDS       = [1, 2]                  # tohum 0 = daha önceki çalıştırma (ft/)
CK_ROOT     = f"{DRIVE_ROOT}/outputs/checkpoints"
FEAT_ROOT   = f"{DRIVE_ROOT}/features"

from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs(CK_ROOT, exist_ok=True); os.makedirs(FEAT_ROOT, exist_ok=True)
assert os.path.exists(CROPS_ZIP), f"crops.zip bulunamadı: {CROPS_ZIP}"
!nvidia-smi -L

In [ ]:
url = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
if os.path.exists(f"/content/{REPO_NAME}"):
    !git -C /content/{REPO_NAME} pull -q
else:
    !git clone -q {url} /content/{REPO_NAME}
%cd /content/{REPO_NAME}
!git log --oneline -1
!pip install -q timm open_clip_torch faiss-cpu pyarrow

In [ ]:
import time
if not os.path.exists(f"{PROCESSED}/crops"):
    t = time.time()
    !cp "{CROPS_ZIP}" /content/crops.zip
    !mkdir -p {PROCESSED} && unzip -q /content/crops.zip -d {PROCESSED} && rm /content/crops.zip
    print(f"açıldı, {time.time()-t:.0f} s")
!ls {PROCESSED}/splits

## Eğitim — her tohum için baştan (uzun hücre, ilerleme satır satır yazılır)

In [ ]:
for seed in SEEDS:
    ck, feat = f"{CK_ROOT}/ft_s{seed}", f"{FEAT_ROOT}/siglip_vitb16+ft_s{seed}"
    print("
################ SEED", seed, "->", ck, "################", flush=True)
    !python scripts/finetune.py --seed {seed} --processed {PROCESSED} --out {ck} --features {feat} --splits val_internal test

## Kontrol ve otomatik kapanış

In [ ]:
import json, glob
for seed in SEEDS:
    h = json.load(open(f"{CK_ROOT}/ft_s{seed}/history.json"))
    print(f"seed {seed}: best val R@1 = {max(r['val_recall@1'] for r in h):.4f} ({len(h)} tur)")
    print("  features:", len(glob.glob(f"{FEAT_ROOT}/siglip_vitb16+ft_s{seed}/*/*_emb.npy")), "dosya")
from google.colab import runtime
runtime.unassign()   # GPU'yu bırakır; sekme açık kalsa da süre harcanmaz